In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy

# 1. Define a simple Global Model to simulate the server's starting point
class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        # A tiny neural network: 10 input features, 2 output classes
        self.fc = nn.Linear(10, 2)

    def forward(self, x):
        return self.fc(x)

# 2. The Client Class (The logic for Assignment 4)
class Client:
    def __init__(self, client_id, model, dataloader, lr=0.01):
        self.id = client_id
        # Client downloads the current global model
        self.local_model = copy.deepcopy(model)
        self.dataloader = dataloader
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.SGD(self.local_model.parameters(), lr=lr)

    def train(self, epochs=1):
        self.local_model.train()
        for epoch in range(epochs):
            for data, target in self.dataloader:
                self.optimizer.zero_grad()
                output = self.local_model(data)
                loss = self.criterion(output, target)
                loss.backward()
                self.optimizer.step()

        # Assignment 4 requirement: Extract the model updates for transmission
        num_samples = len(self.dataloader.dataset)
        return num_samples, self.local_model.state_dict()

# ==========================================
# 3. EXECUTION SCRIPT (This generates the output)
# ==========================================
if __name__ == "__main__":
    print("--- Starting Assignment 4 Simulation ---\n")

    # Initialize the global model
    global_model = SimpleModel()

    # Generate fake local data for the client (100 samples, 10 features)
    X_dummy = torch.randn(100, 10)
    y_dummy = torch.randint(0, 2, (100,))

    # Create a PyTorch DataLoader
    dataset = TensorDataset(X_dummy, y_dummy)
    dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

    # Instantiate Client 1
    client_1 = Client(client_id=1, model=global_model, dataloader=dataloader)

    print(f"Training Client {client_1.id} locally on private data...")
    # Run the local training loop for 3 epochs
    n_samples, updated_weights = client_1.train(epochs=3)

    # --- The Output ---
    print("\n--- Output Generated: Ready for Transmission ---")
    print(f"Data Points Trained On: {n_samples}")
    print("\nExtracted Model Weights (State Dictionary Keys & Shapes):")

    for key, tensor in updated_weights.items():
        print(f"  -> Layer Name: {key} | Tensor Shape: {tensor.shape}")

    print("\nSnippet of the actual updated weight values for 'fc.weight':")
    # We just print the first 5 values of the first row so it doesn't flood your screen
    print(updated_weights['fc.weight'][0][:5])

--- Starting Assignment 4 Simulation ---

Training Client 1 locally on private data...

--- Output Generated: Ready for Transmission ---
Data Points Trained On: 100

Extracted Model Weights (State Dictionary Keys & Shapes):
  -> Layer Name: fc.weight | Tensor Shape: torch.Size([2, 10])
  -> Layer Name: fc.bias | Tensor Shape: torch.Size([2])

Snippet of the actual updated weight values for 'fc.weight':
tensor([ 0.1460,  0.2005, -0.0635, -0.0830, -0.3107])
